# Week 6 - Spark Architecture & Data Processing

This notebook covers Spark's core architecture, lazy evaluation, schema handling,
filtering/selection, column transformations, file format comparisons (CSV vs Parquet),
null handling, and building a simple read -> transform -> filter -> write pipeline.

I'm running this locally with `pyspark` in local mode (`local[*]`), so the Driver and
Executors are just processes on this machine, but the same code runs unchanged on a real
cluster (YARN / Kubernetes / standalone) by just changing the master URL.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("Week6-SparkArchitecture") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark

## Q1. Driver, Cluster Manager, Executor

- **Driver**: the process that runs the `main()`/notebook code, builds the DAG of
  transformations, and turns it into stages/tasks. It also talks back to the Cluster
  Manager and Executors to schedule work and collect results.
- **Cluster Manager**: allocates resources (CPU/memory) across the cluster and hands the
  Driver a set of Executors to run on. Examples: YARN, Kubernetes, Spark Standalone.
- **Executor**: a worker process that actually runs the tasks assigned by the Driver,
  caches data in memory/disk, and reports results back to the Driver.

In `local[*]` mode (used in this notebook), the Driver and Executors run in the same JVM
process on this machine - `*` just tells Spark to use all available cores as worker
threads.

In [ ]:
print("Master:", spark.sparkContext.master)
print("App Name:", spark.sparkContext.appName)
print("Default Parallelism (cores used as 'executors' locally):", spark.sparkContext.defaultParallelism)

Master: local[*]
App Name: Week6-SparkArchitecture
Default Parallelism (cores used as 'executors' locally): 2


## Q2. Lazy Evaluation

Spark doesn't execute a transformation (`select`, `filter`, `withColumn`, etc.) the
moment it's called. Instead it just records it as a node in a **DAG (lineage graph)**.
Nothing actually runs until an **action** (`show`, `count`, `write`, `collect`) is called.

Why this helps performance:
- Spark can look at the *whole* chain of transformations before running anything, and
  optimize it (combine filters, push filters down to the file reader, drop unused
  columns, reorder operations) via the Catalyst optimizer.
- It avoids doing wasted work on data that will be filtered out later anyway.
- It only computes what's actually needed for the action that triggered execution.

Below I chain several transformations - notice nothing actually reads the file yet.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving source.csv to source.csv


In [ ]:
import os
print(os.listdir())

['.config', 'source.csv', 'sample_data']


In [ ]:
df_lazy = spark.read.csv("source.csv", header=True)
df_lazy = df_lazy.select("order_id", "category", "amount")
df_lazy = df_lazy.filter(df_lazy.category == "Electronics")

print(df_lazy.explain())   # shows the planned DAG, no data has been touched yet

== Physical Plan ==
*(1) Filter (isnotnull(category#48) AND (category#48 = Electronics))
+- FileScan csv [order_id#46,category#48,amount#53] Batched: false, DataFilters: [isnotnull(category#48), (category#48 = Electronics)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/source.csv], PartitionFilters: [], PushedFilters: [IsNotNull(category), EqualTo(category,Electronics)], ReadSchema: struct<order_id:string,category:string,amount:string>


None


## Q3. Reading CSV with header + inferSchema

When reading raw CSV, every column comes in as a string by default. Setting
`inferSchema=True` tells Spark to scan the data and guess proper types (Integer, Double,
etc.), and `header=True` tells it to treat the first row as column names instead of data.

In [ ]:
df = spark.read.csv("source.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: integer (nullable = true)

+--------+----------+-----------+--------+------+----------+---------+-------+------+--------+-------+
|order_id|product_id|   category|old_name| price|base_price|   status| amount|region|priority|user_id|
+--------+----------+-----------+--------+------+----------+---------+-------+------+--------+-------+
|       1|      1001|Electronics|  item_1|373.36|     130.0|Completed|2222.59| North|     Low|      1|
|       2|      1002|Electronics|  item_2| 55.91|     124.0|Cancelled| 128.28| South|     Low|      2|
|       3|      1003|       Toys|  item_3|

## Q4. CSV vs Parquet

| | CSV | Parquet |
|---|---|---|
| Storage layout | Row-based - every value of a row sits next to each other | Columnar - values of the same column are stored together |
| Schema | Not stored, has to be inferred or supplied | Stored in the file itself (self-describing) |
| Compression | Poor - text format, generic gzip | Very good - column-level encoding + compression |
| Reading specific columns | Has to read every row in full, then drop columns | Can skip columns entirely on disk |
| Filtering | Has to read all rows then filter in memory | Supports predicate pushdown - skips row groups |

It matters for performance because most analytics queries only touch a handful of
columns and filter on a few conditions. A columnar format like Parquet lets Spark read
only the bytes it actually needs from disk, while CSV forces a full row-by-row scan
every time.

In [ ]:
# write the same data as parquet so we can compare later
import os

csv_file = "/content/source.csv"
parquet_folder = "/content/source_parquet"

csv_size = os.path.getsize(csv_file)

parquet_size = sum(
    os.path.getsize(os.path.join(parquet_folder, f))
    for f in os.listdir(parquet_folder)
    if f.endswith(".parquet")
)

print(f"CSV size:     {csv_size:,} bytes")
print(f"Parquet size: {parquet_size:,} bytes")

CSV size:     148,401 bytes
Parquet size: 68,310 bytes


## Q5. Select specific columns with a filter

Select `product_id` and `price` where `category` is `'Electronics'`.

In [ ]:
electronics_df = df.select("product_id", "price").filter(df.category == "Electronics")
electronics_df.show(5)
print("Row count:", electronics_df.count())

+----------+------+
|product_id| price|
+----------+------+
|      1001|373.36|
|      1002| 55.91|
|      1010|122.23|
|      1015|343.34|
|      1023|122.18|
+----------+------+
only showing top 5 rows
Row count: 404


## Q6. Renaming a column + casting a type

Rename `old_name` -> `new_name`, and cast `price` from String to Double (in our raw
read, `inferSchema` already picked Double for price, so I'll demonstrate the cast
explicitly the way you'd need to if the column came in as a string).

In [ ]:
df_revised = (
    df.withColumnRenamed("old_name", "new_name")
      .withColumn("price", F.col("price").cast(DoubleType()))
)
df_revised.select("new_name", "price").printSchema()
df_revised.select("new_name", "price").show(5)

root
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)

+--------+------+
|new_name| price|
+--------+------+
|  item_1|373.36|
|  item_2| 55.91|
|  item_3|215.56|
|  item_4|146.16|
|  item_5|305.83|
+--------+------+
only showing top 5 rows


## Q7. Lineage Graph (DAG) and fault tolerance

Spark doesn't store the data itself for recovery - it stores the *lineage*: the chain of
transformations that produced each RDD/DataFrame partition, going all the way back to the
original data source. If a worker (Executor) dies and loses some partitions, the Driver
simply looks at the lineage graph and re-runs only the transformations needed to
recompute the lost partitions on a different Executor, instead of restarting the whole
job. This is why Spark doesn't need replicated, mutable storage to be fault tolerant -
the DAG itself is the recovery plan.

## Q8. Filter with AND

Filter `df_orders` where `status == 'Completed'` AND `amount > 1000`.

In [ ]:
df_orders = df
completed_high_value = df_orders.filter((df_orders.status == "Completed") & (df_orders.amount > 1000))
completed_high_value.select("order_id", "status", "amount").show(5)
print("Matching rows:", completed_high_value.count())

+--------+---------+-------+
|order_id|   status| amount|
+--------+---------+-------+
|       1|Completed|2222.59|
|       7|Completed|1171.37|
|      13|Completed|2590.25|
|      19|Completed|2209.13|
|      27|Completed|1357.24|
+--------+---------+-------+
only showing top 5 rows
Matching rows: 437


## Q9. Predicate Pushdown in Parquet

Predicate pushdown means the filter condition (e.g. `amount > 1000`) is passed down to
the Parquet reader itself, not applied after all the data is loaded into memory. Parquet
files store statistics (min/max values) per "row group", so Spark can check those stats
and skip entire row groups that can't possibly match the filter, without decompressing
or reading them at all. The result is that far less data is actually pulled off disk and
into memory compared to reading everything first and filtering afterward in Spark.

In [ ]:
df_parquet = spark.read.parquet("/content/source_parquet")
filtered_parquet = df_parquet.filter(df_parquet.amount > 1000)
filtered_parquet.explain()

== Physical Plan ==
*(1) Filter (isnotnull(amount#250) AND (amount#250 > 1000.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#243,product_id#244,category#245,old_name#246,price#247,base_price#248,status#249,amount#250,region#251,priority#252,user_id#253] Batched: true, DataFilters: [isnotnull(amount#250), (amount#250 > 1000.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/source_parquet], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,1000.0)], ReadSchema: struct<order_id:int,product_id:int,category:string,old_name:string,price:double,base_price:double...




## Q10. Adding a new column

Add `final_price` = `base_price` * 1.18 (18% tax).

In [ ]:
df_with_tax = df.withColumn("final_price", F.round(F.col("base_price") * 1.18, 2))
df_with_tax.select("base_price", "final_price").show(5)

+----------+-----------+
|base_price|final_price|
+----------+-----------+
|     130.0|      153.4|
|     124.0|     146.32|
|    230.11|     271.53|
|     115.5|     136.29|
|    405.49|     478.48|
+----------+-----------+
only showing top 5 rows


## Q11. Transformations vs Actions

- **Transformations** build a new DataFrame from an existing one and are lazy - they
  don't trigger computation. Examples: `select()`, `filter()`, `withColumn()`,
  `groupBy()`, `join()`.
- **Actions** trigger the actual execution of the DAG and return a result to the Driver
  or write data out. Examples: `count()`, `show()`, `collect()`, `write.parquet()`,
  `take()`.

A simple way to remember it: transformations describe *what* to do, actions actually
*do* it.

In [ ]:
# transformation - lazy, nothing executes
transformed = df.filter(df.amount > 500).select("order_id", "amount")

# action - this is what actually triggers a Spark job
print("Row count after action:", transformed.count())

Row count after action: 1674


## Q12. Full pipeline: read Parquet -> filter nulls -> write CSV

In [ ]:
pipeline_df = (
    spark.read.parquet("/content/source_parquet")
        .filter(F.col("user_id").isNotNull())
)

pipeline_df.write.mode("overwrite").option("header", True).csv("data/output_csv")

print("Rows after dropping null user_id:", pipeline_df.count())
pipeline_df.select("order_id", "user_id").show(5)

Rows after dropping null user_id: 1904
+--------+-------+
|order_id|user_id|
+--------+-------+
|       1|      1|
|       2|      2|
|       3|      3|
|       4|      4|
|       5|      5|
+--------+-------+
only showing top 5 rows


## Q13. Client Mode vs Cluster Mode

- **Client Mode**: the Driver runs on the machine where the job was launched (e.g. your
  laptop, an edge node, or this notebook's machine), outside the cluster. The Driver
  talks to the Cluster Manager to get Executors, but if your client machine disconnects
  or crashes, the whole application dies along with it. Common for interactive work
  like notebooks.
- **Cluster Mode**: the Driver itself is launched *inside* the cluster as just another
  managed process, alongside the Executors. The submitting machine can disconnect once
  the job is submitted, and the job keeps running. This is the standard choice for
  production batch jobs submitted via `spark-submit`.

## Q14. Filter with OR

Filter where `region == 'North'` OR `priority == 'High'`.

In [ ]:
north_or_high = df.filter((df.region == "North") | (df.priority == "High"))
north_or_high.select("order_id", "region", "priority").show(5)
print("Matching rows:", north_or_high.count())

+--------+------+--------+
|order_id|region|priority|
+--------+------+--------+
|       1| North|     Low|
|       3| North|    High|
|       4|  West|    High|
|       5| North|  Medium|
|       6| North|    High|
+--------+------+--------+
only showing top 5 rows
Matching rows: 950


## Q15. Why `.show(5)` instead of `.collect()` on a multi-terabyte dataset

`.collect()` pulls *every single row* of the DataFrame back to the Driver's memory as a
Python/Java list. On a multi-terabyte dataset that means trying to cram terabytes of data
into the Driver's RAM, which will almost certainly crash the Driver with an
out-of-memory error long before it finishes - and the whole point of using Spark in the
first place was to avoid that.

`.show(5)` instead asks Spark to compute and return just enough rows to display (5 by
default), and it can often use partial/short-circuited execution to do that without
scanning the entire dataset. It's the safe way to "peek" at data of any size without
risking the Driver.

In [ ]:
df.show(5)   # safe - only materializes a handful of rows on the Driver
# df.collect()  # avoid on large datasets - pulls the ENTIRE dataset into Driver memory

+--------+----------+-----------+--------+------+----------+---------+-------+------+--------+-------+
|order_id|product_id|   category|old_name| price|base_price|   status| amount|region|priority|user_id|
+--------+----------+-----------+--------+------+----------+---------+-------+------+--------+-------+
|       1|      1001|Electronics|  item_1|373.36|     130.0|Completed|2222.59| North|     Low|      1|
|       2|      1002|Electronics|  item_2| 55.91|     124.0|Cancelled| 128.28| South|     Low|      2|
|       3|      1003|       Toys|  item_3|215.56|    230.11|  Pending|2437.82| North|    High|      3|
|       4|      1004|    Grocery|  item_4|146.16|     115.5|  Pending| 351.52|  West|    High|      4|
|       5|      1005|    Grocery|  item_5|305.83|    405.49|Cancelled| 1405.3| North|  Medium|      5|
+--------+----------+-----------+--------+------+----------+---------+-------+------+--------+-------+
only showing top 5 rows


## Wrap-up: brief insights

- Lazy evaluation + the DAG/lineage graph let Spark optimize the *whole* chain of
  transformations before doing any real work, and also give it a cheap way to recover
  from worker failures by recomputing only what was lost.
- Parquet beat CSV on disk size in this test, and `explain()` on the Parquet filter
  showed pushed-down filters - meaning Spark skips data at the file level instead of
  loading everything and filtering in memory.
- Filtering nulls *before* writing kept the output pipeline clean, and using
  `.show()`/`.count()` instead of `.collect()` throughout this notebook is exactly the
  habit you want when the dataset stops being a 2,000-row CSV and starts being a
  multi-terabyte one.

In [ ]:
spark.stop()